# 01 — Prepare data

Set `TARGET_GENE` below to choose the current target.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the project directory whether Jupyter was launched here or from a parent.
PROJECT_DIR = Path.cwd().resolve()
while not (PROJECT_DIR / 'coadread_tcga_pan_can_atlas_2018').exists():
    if PROJECT_DIR.parent == PROJECT_DIR:
        raise FileNotFoundError('Could not find coadread_tcga_pan_can_atlas_2018.')
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / 'coadread_tcga_pan_can_atlas_2018'
MRNA_PATH = DATA_DIR / 'data_mrna_seq_v2_rsem.txt'
METHYLATION_PATH = DATA_DIR / 'data_methylation_hm450.txt'
CNA_PATH = DATA_DIR / 'data_log2_cna.txt'

TARGET_GENE = 'EREG'  # Change this to choose the RNA-expression target.
PROJECT_DIR

PosixPath('/Users/davidstone/Documents/School/Coursework/Classes/2026 Fall/Machine Learning/Project')

## Define the shared sample cohort from the matrix headers

The cohort is calculated from the matrix columns rather than assumed from a case-list description. We use methylation-file order as the canonical order, then explicitly reindex RNA expression to it. This protects against the same samples being present in different column orders.

In [2]:
def sample_columns(path, metadata_columns):
    columns = pd.read_csv(path, sep='\t', nrows=0).columns.tolist()
    samples = [column for column in columns if column not in metadata_columns]
    if len(samples) != len(set(samples)):
        raise ValueError(f'Duplicate sample labels in {path.name}')
    return samples

mrna_samples = sample_columns(MRNA_PATH, {'Hugo_Symbol', 'Entrez_Gene_Id'})
methylation_samples = sample_columns(
    METHYLATION_PATH,
    {'ENTITY_STABLE_ID', 'NAME', 'DESCRIPTION', 'TRANSCRIPT_ID'},
)
cna_samples = sample_columns(CNA_PATH, {'Hugo_Symbol', 'Entrez_Gene_Id'})

shared_samples = [
    sample for sample in methylation_samples
    if sample in set(mrna_samples) and sample in set(cna_samples)
]

cohort_summary = pd.Series({
    'mRNA samples': len(mrna_samples),
    'methylation samples': len(methylation_samples),
    'CNA samples': len(cna_samples),
    'shared samples': len(shared_samples),
})
display(cohort_summary)

# The current checked data files contain 379 unique shared samples.
# Do not replace this with 382: the methylation case list has duplicate entries.
assert len(shared_samples) == 379, 'Unexpected shared cohort size; inspect the source files.'
assert len(shared_samples) == len(set(shared_samples))

mRNA samples           592
methylation samples    379
CNA samples            592
shared samples         379
dtype: int64

## Create and output test manifest

The manifest is created once from a fixed seed. The remaining training samples will later be used for all cross-validation, preprocessing fitting, tuning, and model selection.

In [3]:
SPLIT_SEED = 20261121
FINAL_TEST_FRACTION = 0.20
SPLIT_DIR = PROJECT_DIR / 'results' / 'splits'
FINAL_HOLDOUT_MANIFEST = SPLIT_DIR / 'train_test_manifest.csv'

if FINAL_HOLDOUT_MANIFEST.exists():
    split_manifest = pd.read_csv(FINAL_HOLDOUT_MANIFEST)
    required_columns = {'sample_id', 'analysis_role'}
    if not required_columns.issubset(split_manifest.columns):
        raise ValueError('Existing holdout manifest has unexpected columns.')
    if split_manifest['sample_id'].tolist() != shared_samples:
        raise ValueError('Existing holdout manifest does not match this shared cohort.')
    if set(split_manifest['analysis_role']) != {'train', 'test'}:
        raise ValueError('Holdout manifest must contain train and test roles.')
else:
    n_test = int(np.ceil(FINAL_TEST_FRACTION * len(shared_samples)))
    rng = np.random.default_rng(SPLIT_SEED)
    test_samples = set(rng.permutation(shared_samples)[:n_test])
    split_manifest = pd.DataFrame({'sample_id': shared_samples})
    split_manifest['analysis_role'] = np.where(
        split_manifest['sample_id'].isin(test_samples), 'test', 'train'
    )
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    split_manifest.to_csv(FINAL_HOLDOUT_MANIFEST, index=False)

train_sample_ids = split_manifest.loc[
    split_manifest['analysis_role'].eq('train'), 'sample_id'
].tolist()
test_sample_ids = split_manifest.loc[
    split_manifest['analysis_role'].eq('test'), 'sample_id'
].tolist()

assert set(train_sample_ids).isdisjoint(test_sample_ids)
assert len(train_sample_ids) + len(test_sample_ids) == len(shared_samples)
display(split_manifest['analysis_role'].value_counts())
print(f'Holdout manifest: {FINAL_HOLDOUT_MANIFEST}')

analysis_role
train    303
test      76
Name: count, dtype: int64

Holdout manifest: /Users/davidstone/Documents/School/Coursework/Classes/2026 Fall/Machine Learning/Project/results/splits/train_test_manifest.csv
